# PKTD3-TD: Real-Time Checkpoint & Actor Saturation Monitor

This independent diagnostic notebook allows you to monitor training checkpoints in real time as they land on **Google Drive** without interrupting or touching the running `train_colab.ipynb` training kernel.

### How to Use:
1. Open this notebook in a **separate Colab tab** while `train_colab.ipynb` is running.
2. Mount the same Google Drive.
3. Run the diff check cell at any time (e.g. after episode 500, 1000, 1500, etc.).
4. Verify that `mean_abs_diff > 0.01` and the actor is actively learning rather than frozen.

In [ ]:
# Step 1: Mount Google Drive to access ongoing checkpoints
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Step 2: Clone repository and install package (lightweight, dev dependencies omitted)
!git clone https://github.com/Krishna200608/uav_trajectory_rl.git
%cd uav_trajectory_rl
!pip install -e . --quiet

In [ ]:
# Step 3: Set the Drive checkpoint path where training is actively writing
# Default path matching train_colab.ipynb Drive backup directory:
CHECKPOINT_DIR = "/content/drive/MyDrive/Uav_trajectory_rl/PKTD3_TD_Checkpoints/run2"

# Confirm existing checkpoint files
!ls -la "{CHECKPOINT_DIR}"

In [ ]:
# Step 4: Evaluate actor outputs on fixed test states across checkpoints
import glob, re, torch
import numpy as np
from uav_trajectory_rl.td3_networks import Actor

# Find all checkpoints currently on disk, sorted by episode number
ckpt_files = glob.glob(f"{CHECKPOINT_DIR}/td3_agent_ep*.pt")
episodes = sorted(int(re.search(r"ep(\d+)", f).group(1)) for f in ckpt_files)
print("Checkpoints found so far:", episodes)

if not episodes:
    print("No episode checkpoints found yet. Re-run this cell once the first checkpoint (e.g. ep500) has saved.")
else:
    # Fixed test states -- same seed every time you run this, so results are comparable
    # across different points in training and across different runs
    rng = np.random.default_rng(999)
    test_states = rng.uniform(-1.0, 1.0, size=(8, 26)).astype(np.float32)  # state_dim=26 for K=10
    states_t = torch.as_tensor(test_states)

    results = {}
    for ep in episodes:
        ckpt = torch.load(f"{CHECKPOINT_DIR}/td3_agent_ep{ep}.pt", map_location="cpu", weights_only=True)
        actor = Actor(state_dim=26, action_dim=3, max_action=1.0)
        actor.load_state_dict(ckpt["actor"])
        actor.eval()
        with torch.no_grad():
            out = actor(states_t).numpy()
        results[ep] = out
        print(f"\n--- ep{ep} (total_updates={ckpt['total_updates']}) ---")
        print(out.round(4))

    print("\n=== Mean abs change between consecutive checkpoints ===")
    for a, b in zip(episodes, episodes[1:]):
        diff = np.abs(results[b] - results[a]).mean()
        saturated_frac = (np.abs(results[b]) > 0.999).mean()
        print(f"ep{a} -> ep{b}: mean_abs_diff={diff:.6f}, frac_outputs_saturated(|x|>0.999)={saturated_frac:.2f}")

    print("\n=== INTERPRETATION ===")
    last_diff = np.abs(results[episodes[-1]] - results[episodes[-2]]).mean() if len(episodes) > 1 else None
    if last_diff is not None and last_diff < 1e-4:
        print("WARNING: latest two checkpoints are near-identical -- actor may be frozen again. Consider stopping and investigating before letting the full run finish.")
    else:
        print("OK: actor output is still changing between checkpoints -- training appears active, not frozen.")